### Optional: Train VirtNiche

Training VirtNiche is optional. To reproduce the downstream analyses directly, you can skip this notebook and download our pretrained model from Zenodo (DOI: `10.5281/zenodo.22760559`).

If you run `3.VirtNiche_train.ipynb`, the trained model will be saved to:

`./VirtNiche_Perturb_FISH_test_model/`

Alternatively, place the downloaded pretrained model in the same location before running `4.Virtual_Niche_Analysis.ipynb`.

In [2]:
import sys
import warnings
from pathlib import Path

import scanpy as sc

warnings.simplefilter("ignore", UserWarning)

sys.path.insert(0, str(Path("..").resolve()))

import VirtNiche
from VirtNiche._utils import compute_spatial_neighbors

Global seed set to 0
/home/sirm/software/miniconda3/envs/VirtNiche/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
adata_path = "./Perturb-FISH-rawadata/Measured_annotated_splited.h5ad"

result_name = "Perturb_FISH_test"

k_neighbors = 6
seed = 42

max_epochs = 500
batch_size = 512
early_stopping_patience = 15

ordered_keys = ["perturb_gene"]
categorical_keys = ["cell_type"]

In [3]:
adata = sc.read(adata_path)

adata

AnnData object with n_obs × n_vars = 120594 × 500
    obs: 'area_of_the_cell_in_pixels', 'gRNA', 'perturb', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'leiden', 'manual', 'cell_type', 'perturb_celltype_key', 'microenv_key', 'celltype_microenv_key', 'niche_key', 'split'
    var: 'id', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts'
    obsm: 'perturb_gene', 'spatial'

In [4]:
neighbors_index = compute_spatial_neighbors(
    adata,
    n_neighbors=k_neighbors,
    spatial_key="spatial",
)

neighbors_index.shape

(120594, 6)

In [5]:
VirtNiche.VirtNiche.setup_anndata(
    adata,
    ordered_attributes_keys=ordered_keys,
    categorical_attributes_keys=categorical_keys,
)

INFO     Generating sequential column names                                                                        


In [6]:
module_params = {
    "decoder_width": 1024,
    "decoder_depth": 4,
    "attribute_nn_width": 512,
    "attribute_nn_depth": 2,
    "n_latent_attribute_categorical": 4,
    "gene_likelihood": "nb",
    "reconstruction_penalty": 1e2,
    "unknown_attribute_penalty": 1e1,
    "unknown_attribute_noise_param": 1e-1,
    "attribute_dropout_rate": 0.1,
    "use_batch_norm": False,
    "use_layer_norm": False,
    "seed": seed,
    "unknown_attributes": False,
    "eval_r2_ordered": True,
    "neighbors_index": neighbors_index,
    "agg_mode": "gat",
    "sample": False,
}

In [7]:
model = VirtNiche.VirtNiche(
    adata=adata,
    n_latent=32,
    model_name=result_name,
    module_params=module_params,
    split_key="split",
    train_split="train",
    valid_split="val",
    test_split="ood",
)

Global seed set to 42


In [8]:
trainer_params = {
    "n_epochs_warmup": 0,
    "latent_lr": 0.0001,
    "latent_wd": 0.0001,
    "niche_lr": 0.0001,
    "niche_wd": 0.0001,
    "decoder_lr": 0.0001,
    "decoder_wd": 0.0001,
    "attribute_nn_lr": 0.01,
    "attribute_nn_wd": 0.00000004,
    "step_size_lr": 45,
    "cosine_scheduler": True,
    "scheduler_final_lr": 0.00001,
}

trainer_params

{'n_epochs_warmup': 0,
 'latent_lr': 0.0001,
 'latent_wd': 0.0001,
 'niche_lr': 0.0001,
 'niche_wd': 0.0001,
 'decoder_lr': 0.0001,
 'decoder_wd': 0.0001,
 'attribute_nn_lr': 0.01,
 'attribute_nn_wd': 4e-08,
 'step_size_lr': 45,
 'cosine_scheduler': True,
 'scheduler_final_lr': 1e-05}

In [9]:
model.train(
    max_epochs=max_epochs,
    batch_size=batch_size,
    monitor="val_neg_total_loss",
    plan_kwargs=trainer_params,
    early_stopping=True,
    early_stopping_patience=early_stopping_patience,
    check_val_every_n_epoch=10,
    num_workers=1,
    enable_checkpointing=False,
)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA GeForce RTX 3090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2]


Epoch 54/500:  11%|█         | 54/500 [37:27<5:09:23, 41.62s/it, v_num=1, val_generative_mean_accuracy=0.935, val_generative_var_accuracy=0.505, val_VirtNiche_metric=0.72, val_reconstruction_loss=415, val_unknown_attribute_penalty_loss=0, val_cell_r2_mean=0.679, val_cell_pearson_mean=0.842, val_cell_spearman_mean=0.39, val_neg_total_loss=-415, generative_mean_accuracy=0, generative_var_accuracy=0, VirtNiche_metric=0, reconstruction_loss=415, unknown_attribute_penalty_loss=0, cell_r2_mean=0, cell_pearson_mean=0, cell_spearman_mean=0, neg_total_loss=-415]  
Monitored metric val_neg_total_loss did not improve in the last 15 records. Best score: -414.971. Signaling Trainer to stop.


In [10]:
model.save()